In [1]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import ConfigurableField

load_dotenv()
llm = ChatOpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL_NAME"),
    base_url=os.getenv("OPENAI_API_BASE"),
    temperature=0
    )

## Chain & Pipe

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("tell me a joke about {topic}")
chain = prompt | llm | StrOutputParser()

In [18]:
chain.invoke({"topic":"chicken"})

"Sure! Here's a classic chicken joke for you:\n\nWhy did the chicken go to the seance?\n\nTo talk to the other side! 🐔✨"

In [19]:
analysis_prompt = ChatPromptTemplate.from_template("is this a funny joke? {joke}")

composed_chain = {"joke": chain} | analysis_prompt | llm | StrOutputParser()

composed_chain.invoke({"topic": "bears"})

"Yes, that's a funny joke! It's a clever play on words, combining the idea of a polar bear (which lives in cold, icy environments) with the concept of something dissolving in water. The unexpected twist makes it humorous! 🐻❄️"

In [21]:
analysis_prompt = ChatPromptTemplate.from_template("is this a funny joke? {joke}")
composed_chain_with_lambda = (
    chain
    | (lambda input: {"joke": input}) # 作为analysis_prompt的输入
    | analysis_prompt
    | llm
    | StrOutputParser()
)

composed_chain_with_lambda.invoke({"topic": "beets"})

'Yes, that\'s a classic pun-based joke, and it can definitely be funny, especially if the audience appreciates wordplay and lighthearted humor! The humor comes from the double meaning of "dressing" (both as salad dressing and the idea of getting dressed). If you\'re sharing it with someone who enjoys puns or vegetable-themed jokes, it should get a chuckle! 😄'

In [11]:
from langchain_core.runnables import RunnableParallel

composed_chain_with_pipe = (
    RunnableParallel({"joke": chain})
    .pipe(analysis_prompt)
    .pipe(llm)
    .pipe(StrOutputParser())
)

composed_chain_with_pipe.invoke({"topic": "battlestar galactica"})

'Yes, that\'s a funny joke! It plays on the central theme of identity in *Battlestar Galactica*, where the Cylons are constantly grappling with their own nature and the blurred lines between human and machine. The pun on "issues" works well, especially for fans of the show who are familiar with the existential struggles of the Cylons. It’s clever and lighthearted—great for a chuckle! 😄'

## Formatting with RunnableParallels

RunnableParallels are useful for parallelizing operations, but can also be useful for manipulating the output of one Runnable to match the input format of the next Runnable in a sequence. You can use them to split or fork the chain so that multiple components can process the input in parallel. Later, other components can join or merge the results to synthesize a final response. This type of chain creates a computation graph that looks like the following:

       Input
        / \
       /   \
 Branch1 Branch2

       \   /
        \ /
        Combine

 {"context": retriever, "question": RunnablePassthrough()}

RunnableParallel({"context": retriever, "question": RunnablePassthrough()})

RunnableParallel(context=retriever, question=RunnablePassthrough())

In [23]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings

vectorstore = FAISS.from_texts(
    ["harrison worked at kensho"], embedding=HuggingFaceEmbeddings()
)
retriever = vectorstore.as_retriever()
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# The prompt expects input with keys for "context" and "question"
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(api_key="sk-762684b96deb4f748cb4383757f69a09",model="deepseek-chat" ,base_url="https://api.deepseek.com")

retrieval_chain = (
    {"context": retriever, "question": RunnablePassthrough()} # get the context using our retriever and passthrough the user input under the "question" key
    | prompt #
    | model
    | StrOutputParser()
)

retrieval_chain.invoke("where did harrison work?")

'Harrison worked at Kensho.'

## Using itemgetter as shorthand

Note that you can use Python's itemgetter as shorthand to extract data from the map when combining with RunnableParallel. You can find more information about itemgetter in the Python Documentation.

In [ ]:
from operator import itemgetter

chain = (
    {
        "context": itemgetter("question") | retriever,
        "question": itemgetter("question"),
        "language": itemgetter("language"),  # itemgetter是根据字典查找输入中的对应内容， RunnalbePassthrough()是直接传递输入
    }
    | prompt
    | model
    | StrOutputParser()
)

chain.invoke({"question": "where did harrison work", "language": "italian"})

'Based on the provided context, Harrison worked at Kensho.'

## Parallelize steps

RunnableParallels make it easy to execute multiple Runnables in parallel, and to return the output of these Runnables as a map.##

In [14]:
joke_chain = ChatPromptTemplate.from_template("tell me a joke about {topic}") | model
poem_chain = (
    ChatPromptTemplate.from_template("write a 2-line poem about {topic}") | model
)

map_chain = RunnableParallel(joke=joke_chain, poem=poem_chain)

map_chain.invoke({"topic": "bear"})

{'joke': AIMessage(content="Sure, here's a bear joke for you:\n\nWhy did the bear dissolve in water?\n\nBecause it was a polar bear! 🐻❄️", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 9, 'total_tokens': 40, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 9}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-7161adea-79fc-4975-87f8-281122df16f9-0', usage_metadata={'input_tokens': 9, 'output_tokens': 31, 'total_tokens': 40, 'input_token_details': {}, 'output_token_details': {}}),
 'poem': AIMessage(content="In forest deep, the bear does roam,  \nA gentle giant, nature's home.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 11, 'total_tokens': 29, 'completion_tokens_details': None, 'prompt_tokens_details': No

## Parallelism

RunnableParallel are also useful for running independent processes in parallel, since each Runnable in the map is executed in parallel. For example, we can see our earlier joke_chain, poem_chain and map_chain all have about the same runtime, even though map_chain executes both of the other two.

In [11]:
%%timeit

joke_chain.invoke({"topic": "bear"})

1.82 s ± 147 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [12]:
%%timeit

poem_chain.invoke({"topic": "bear"})

1.39 s ± 125 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit

map_chain.invoke({"topic": "bear"})

1.68 s ± 177 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## add default invocation args to a Runnable
Sometimes we want to invoke a Runnable within a RunnableSequence with constant arguments that are not part of the output of the preceding Runnable in the sequence, and which are not part of the user input. We can use the Runnable.bind() method to set these arguments ahead of time.




In [ ]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write out the following equation using algebraic symbols then solve it. Use the format\n\nEQUATION:...\nSOLUTION:...\n\n",
        ), # prompt for system
        ("human", "{equation_statement}"), # prompt for human
    ]
)

#method 1

runnable = (
    {"equation_statement": RunnablePassthrough()} | prompt | model | StrOutputParser() 
)

print(runnable.invoke("x raised to the third plus seven equals 12"))



EQUATION: \( x^3 + 7 = 12 \)

SOLUTION:
1. Subtract 7 from both sides:
   \[
   x^3 = 12 - 7
   \]
   \[
   x^3 = 5
   \]
2. Take the cube root of both sides:
   \[
   x = \sqrt[3]{5}
   \]
   \[
   x \approx 1.71
   \]

So, the solution is \( x = \sqrt[3]{5} \) or approximately \( x \approx 1.71 \).


In [25]:
#method 2
runnable = (prompt | model | StrOutputParser()) 
print(runnable.invoke({"equation_statement": "x raised to the third plus seven equals 12"}))

EQUATION: \( x^3 + 7 = 12 \)

SOLUTION:
1. Subtract 7 from both sides:
   \[
   x^3 = 12 - 7
   \]
   \[
   x^3 = 5
   \]
2. Take the cube root of both sides:
   \[
   x = \sqrt[3]{5}
   \]

So, the solution is \( x = \sqrt[3]{5} \).


In [16]:
runnable = (
    {"equation_statement": RunnablePassthrough()}
    | prompt
    | model.bind(stop="SOLUTION") #model.bind() add other tuntime args.
    | StrOutputParser()
)

print(runnable.invoke("x raised to the third plus seven equals 12"))

EQUATION: \( x^3 + 7 = 12 \)




## Using the RunnableLambda constructor 

wrap our custom logic using the RunnableLambda constructor.

In [26]:
from langchain_core.runnables import RunnableLambda

def length_function(text):
    return len(text)

def _multiple_length_function(text1, text2):
    return len(text1) * len(text2)

def multiple_length_function(_dict):
    return _multiple_length_function(_dict["text1"], _dict["text2"])


prompt = ChatPromptTemplate.from_template("what is {a} + {b}")
chain1 = prompt | model

chain = (
    {
        "a": itemgetter("foo") | RunnableLambda(length_function), # single input lambda
        "b": {"text1": itemgetter("foo"), "text2": itemgetter("bar")} # multiple input lambda
        | RunnableLambda(multiple_length_function),
    }
    | prompt
    | model
)

chain.invoke({"foo": "bar", "bar": "gah"})

AIMessage(content='**Solution:**\n\nTo find the sum of 3 and 9, follow these steps:\n\n1. **Identify the numbers to add:**\n   \n   \\[\n   3 \\quad \\text{and} \\quad 9\n   \\]\n\n2. **Add the numbers together:**\n   \n   \\[\n   3 + 9 = 12\n   \\]\n\n3. **Final Answer:**\n   \n   \\[\n   \\boxed{12}\n   \\]', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 95, 'prompt_tokens': 10, 'total_tokens': 105, 'completion_tokens_details': None, 'prompt_tokens_details': None, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 10}, 'model_name': 'deepseek-chat', 'system_fingerprint': 'fp_3a5770e1b4', 'finish_reason': 'stop', 'logprobs': None}, id='run-b398159d-3b4c-42e4-9c86-86f791cd959c-0', usage_metadata={'input_tokens': 10, 'output_tokens': 95, 'total_tokens': 105, 'input_token_details': {}, 'output_token_details': {}})

## The convenience @chain decorator

You can also turn an arbitrary function into a chain by adding a @chain decorator. This is functionally equivalent to wrapping the function in a RunnableLambda constructor as shown above. Here's an example

In [28]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import chain

prompt1 = ChatPromptTemplate.from_template("Tell me a joke about {topic}")
prompt2 = ChatPromptTemplate.from_template("What is the subject of this joke: {joke}")


@chain
def custom_chain(text):
    prompt_val1 = prompt1.invoke({"topic": text})
    output1 = llm.invoke(prompt_val1)
    parsed_output1 = StrOutputParser().invoke(output1)
    chain2 = prompt2 | llm | StrOutputParser()
    return chain2.invoke({"joke": parsed_output1})


custom_chain.invoke("bears")

'The subject of the joke is **bears**. The humor plays on the double meaning of "bear feet" (bare feet), which is a pun related to bears not wearing shoes. 🐻👣'

## Automatic coercion in chains

When using custom functions in chains with the pipe operator (|), you can omit the RunnableLambda or @chain constructor and rely on coercion. Here's a simple example with a function that takes the output from the model and returns the first five letters of it:

In [29]:
prompt = ChatPromptTemplate.from_template("tell me a story about {topic}")

chain_with_coerced_function = prompt | model | (lambda x: x.content[:5])

chain_with_coerced_function.invoke({"topic": "bears"})

'Once '

## Passing run metadata

Runnable lambdas can optionally accept a RunnableConfig parameter, which they can use to pass callbacks, tags, and other configuration information to nested runs.

In [30]:
import json

from langchain_core.runnables import RunnableConfig


def parse_or_fix(text: str, config: RunnableConfig):
    fixing_chain = (
        ChatPromptTemplate.from_template(
            "Fix the following text:\n\n\`\`\`text\n{input}\n\`\`\`\nError: {error}"
            " Don't narrate, just respond with the fixed data."
        )
        | model
        | StrOutputParser()
    )
    for _ in range(3):
        try:
            return json.loads(text)
        except Exception as e:
            text = fixing_chain.invoke({"input": text, "error": e}, config)
    return "Failed to parse"


from langchain_community.callbacks import get_openai_callback

with get_openai_callback() as cb:
    output = RunnableLambda(parse_or_fix).invoke(
        "{foo: bar}", {"tags": ["my-tag"], "callbacks": [cb]}
    )
    print(output)
    print(cb)

<>:9: SyntaxWarning: invalid escape sequence '\`'
<>:9: SyntaxWarning: invalid escape sequence '\`'
/var/folders/fz/gqgl22xd2x32v6cc7mt6pjj40000gn/T/ipykernel_59986/1261273371.py:9: SyntaxWarning: invalid escape sequence '\`'
  "Fix the following text:\n\n\`\`\`text\n{input}\n\`\`\`\nError: {error}"


Failed to parse
Tokens Used: 213
	Prompt Tokens: 183
		Prompt Tokens Cached: 0
	Completion Tokens: 30
		Reasoning Tokens: 0
Successful Requests: 3
Total Cost (USD): $0.0


## Streaming

RunnableLambda is best suited for code that does not need to support streaming. If you need to support streaming (i.e., be able to operate on chunks of inputs and yield chunks of outputs), use RunnableGenerator instead as in the example below.

You can use generator functions (ie. functions that use the yield keyword, and behave like iterators) in a chain.

The signature of these generators should be Iterator[Input] -> Iterator[Output]. Or for async generators: AsyncIterator[Input] -> AsyncIterator[Output].

These are useful for:

implementing a custom output parser
modifying the output of a previous step, while preserving streaming capabilities
Here's an example of a custom output parser for comma-separated lists. First, we create a chain that generates such a list as text:


In [34]:
# without streaming

prompt = ChatPromptTemplate.from_template(
    "Write a comma-separated list of 5 animals similar to: {animal}. Do not include numbers"
)

str_chain = prompt | model | StrOutputParser()

str_chain.invoke({"animal": "bear"})

'bear, panda, koala, raccoon, wolverine'

In [35]:
from typing import Iterator, List

prompt = ChatPromptTemplate.from_template(
    "Write a comma-separated list of 5 animals similar to: {animal}. Do not include numbers"
)

str_chain = prompt | model | StrOutputParser()

for chunk in str_chain.stream({"animal": "bear"}):   # .stream()返回一个迭代器
    print(chunk, end="", flush=True)

bear, panda, koala, raccoon, wolverine

In [39]:
# This is a custom parser that splits an iterator of llm tokens
# into a list of strings separated by commas
def split_into_list(input: Iterator[str]) -> Iterator[List[str]]:
    # hold partial input until we get a comma
    buffer = ""
    for chunk in input:
        # add current chunk to buffer
        buffer += chunk
        # while there are commas in the buffer
        while "," in buffer:
            # split buffer on comma
            comma_index = buffer.index(",")
            # yield everything before the comma
            yield [buffer[:comma_index].strip()]
            # save the rest for the next iteration
            buffer = buffer[comma_index + 1 :]
    # yield the last chunk
    yield [buffer.strip()]


list_chain = str_chain | split_into_list

for chunk in list_chain.stream({"animal": "bear"}):
    print(chunk, flush=True)

['bear']
['panda']
['koala']
['raccoon']
['wolverine']


In [40]:
list_chain.invoke({"animal": "bear"})   

['bear', 'panda', 'koala', 'raccoon', 'wolverine']

In [41]:
## Ansync version of the above code

from typing import AsyncIterator


async def asplit_into_list(
    input: AsyncIterator[str],
) -> AsyncIterator[List[str]]:  # async def
    buffer = ""
    async for (
        chunk
    ) in input:  # `input` is a `async_generator` object, so use `async for`
        buffer += chunk
        while "," in buffer:
            comma_index = buffer.index(",")
            yield [buffer[:comma_index].strip()]
            buffer = buffer[comma_index + 1 :]
    yield [buffer.strip()]


list_chain = str_chain | asplit_into_list

async for chunk in list_chain.astream({"animal": "bear"}):
    print(chunk, flush=True)

['bear']
['panda']
['koala']
['raccoon']
['wolverine']
